In [75]:
import os
import numpy as np
import matplotlib.pyplot as plt
import flopy
from matplotlib import cm
import matplotlib.colors as mcolors
import matplotlib.animation as animation

In [ ]:
# fpths to data
mf6_model_ws = os.path.join('model', 'reactive')
mf6_name = "gwf"
mf96_base_dir = os.path.join('data', 'heads')
output_dir = "headscomparison"
os.makedirs(output_dir, exist_ok=True)

In [71]:
# model info
nlay, nrow, ncol = 12, 10, 51
nper = 39

# Period data (nstp, tsmult, steady)
perioddata = [
    (2, 2, 1), (4, 4, 1), (4, 4, 1), (4, 4, 1), (7, 7, 1),
    (7, 7, 1), (7, 7, 1), (7, 7, 1), (14, 14, 1), (14, 14, 1), 
    (15, 15, 1), (13, 13, 1), (14, 14, 1), (14, 14, 1), (14, 14, 1), 
    (21, 21, 1), (35, 35, 1), (28, 28, 1), (28, 28, 1), (28, 28, 1), 
    (28, 28, 1), (28, 28, 1), (28, 28, 1), (28, 28, 1), (28, 28, 1), 
    (35, 35, 1), (35, 35, 1), (28, 28, 1), (28, 28, 1), (28, 28, 1), 
    (35, 35, 1), (35, 35, 1), (28, 28, 1), (28, 28, 1), (28, 28, 1), 
    (28, 28, 1), (35, 35, 1), (35, 35, 1), (28, 28, 1)
]

In [72]:
def read_ascii_wrapped_head(file_path, shape):
    with open(file_path, 'r') as f:
        f.readline()  # skip header
        data = []
        for line in f:
            for val in line.strip().split():
                try:
                    data.append(float(val))
                except ValueError:
                    continue
    arr = np.array(data)
    if arr.size != shape[0] * shape[1]:
        raise ValueError(f"Unexpected data size in {file_path}, got {arr.size}, expected {shape[0]*shape[1]}")
    
    # Flip rows vertically to match MF6 orientation
    return arr.reshape(shape)

In [73]:
# Load MF6 simulation and head file
sim = flopy.mf6.MFSimulation.load(sim_ws=mf6_model_ws)
gwf = sim.get_model(mf6_name)
head_obj = flopy.utils.HeadFile(os.path.join(mf6_model_ws, f"{mf6_name}.hds"))

loading simulation...
  loading simulation name file...
  loading tdis package...
  loading model gwf6...
    loading package dis...
    loading package ic...
    loading package npf...
    loading package sto...
    loading package chd...
    loading package wel...
    loading package wel...
    loading package oc...
  loading model gwt6...
    loading package dis...
    loading package ic...
    loading package adv...
    loading package dsp...
    loading package ssm...
    loading package mst...
    loading package oc...
    loading package obs...
  loading model gwt6...
    loading package dis...
    loading package ic...
    loading package adv...
    loading package dsp...
    loading package ssm...
    loading package mst...
    loading package oc...
    loading package obs...
  loading model gwt6...
    loading package dis...
    loading package ic...
    loading package adv...
    loading package dsp...
    loading package ssm...
    loading package mst...
    loading package

In [ ]:
# Build list of (kstp, kper) pairs
kstpkper_list = []
for kper, (nstp, _, _) in enumerate(perioddata):
    if kper in [0, 1]:  # First and last timestep for SP 1 and 2
        kstpkper_list.append((0, kper))
        kstpkper_list.append((nstp - 1, kper))
    else:  # Only last timestep for others
        kstpkper_list.append((nstp - 1, kper))

# Compare and plot 
for i, (kstp, kper) in enumerate(kstpkper_list):
    sp_num = kper + 1
    ts_num = kstp + 1
    folder_id = f"{i:03d}_sp{sp_num:02d}_ts{ts_num:02d}"
    mf96_folder = os.path.join(mf96_base_dir, folder_id)

    if not os.path.exists(mf96_folder):
        print(f"[Missing Folder] {mf96_folder}")
        continue

    mf6_heads = head_obj.get_data(kstpkper=(kstp, kper))  # shape: (12, 10, 51)

    for lay in range(nlay):
        mf6_layer = mf6_heads[lay, :, :]  # (10, 51)
        mf96_file = f"layer_{lay+1}_stressperiod_{sp_num}_timestep_{ts_num}"
        mf96_path = os.path.join(mf96_folder, mf96_file)

        if not os.path.exists(mf96_path):
            print(f"[Missing File] {mf96_path}")
            continue

        try:
            mf96_layer = read_ascii_wrapped_head(mf96_path, (nrow, ncol))
        except Exception as e:
            print(f"[Read Error] {mf96_path}: {e}")
            continue

        nodata = -999.99
        valid_mask = mf96_layer != nodata
        if not np.any(valid_mask):
            print(f"[No Valid Data] {mf96_path}")
            continue

        diff = mf6_layer[valid_mask] - mf96_layer[valid_mask]
        rmse = np.sqrt(np.mean(diff**2))
        max_diff = np.abs(diff).max()

        print(f"SP {sp_num:02d} TS {ts_num:02d} Layer {lay+1:02d} → RMSE: {rmse:.4f}, Max Diff: {max_diff:.4f}")

        # --- Plot ---
        diff_map = np.full_like(mf96_layer, np.nan)
        diff_map[valid_mask] = diff

        plt.figure(figsize=(8, 5))
        cmap = plt.get_cmap("RdBu")  # Regular RdBu: red = negative, blue = positive
        bounds = np.arange(-1.0, 1.01, 0.1)
        norm = mcolors.BoundaryNorm(bounds, cmap.N)

        im = plt.imshow(diff_map, cmap=cmap, norm=norm)
        plt.colorbar(im, boundaries=bounds, ticks=bounds, extend='both', label="Head Diff (m)")
        plt.title(f"Head Difference (MF6 - MF96)\nSP {sp_num:02d} TS {ts_num:02d} Layer {lay+1:02d}")
        plt.xlabel("Column Index")
        plt.ylabel("Row Index")
        plt.tight_layout()

        png_file = f"sp{sp_num:02d}_ts{ts_num:02d}_layer{lay+1:02d}.png"
        plt.savefig(os.path.join(output_dir, png_file), dpi=300)
        plt.close()

SP 01 TS 01 Layer 01 → RMSE: 0.0770, Max Diff: 0.2409
SP 01 TS 01 Layer 02 → RMSE: 0.0214, Max Diff: 0.0405
SP 01 TS 01 Layer 03 → RMSE: 0.0250, Max Diff: 0.0760
SP 01 TS 01 Layer 04 → RMSE: 0.0295, Max Diff: 0.0937
SP 01 TS 01 Layer 05 → RMSE: 0.0324, Max Diff: 0.1209
SP 01 TS 01 Layer 06 → RMSE: 0.0507, Max Diff: 0.2607
SP 01 TS 01 Layer 07 → RMSE: 0.0572, Max Diff: 0.4302
SP 01 TS 01 Layer 08 → RMSE: 0.1881, Max Diff: 0.5972
SP 01 TS 01 Layer 09 → RMSE: 0.0572, Max Diff: 0.1523
SP 01 TS 01 Layer 10 → RMSE: 0.0098, Max Diff: 0.0162
SP 01 TS 01 Layer 11 → RMSE: 0.0054, Max Diff: 0.0080
SP 01 TS 01 Layer 12 → RMSE: 0.0016, Max Diff: 0.0018
SP 01 TS 02 Layer 01 → RMSE: 0.0880, Max Diff: 0.2650
SP 01 TS 02 Layer 02 → RMSE: 0.0307, Max Diff: 0.0546
SP 01 TS 02 Layer 03 → RMSE: 0.0338, Max Diff: 0.0898
SP 01 TS 02 Layer 04 → RMSE: 0.0375, Max Diff: 0.1075
SP 01 TS 02 Layer 05 → RMSE: 0.0398, Max Diff: 0.1336
SP 01 TS 02 Layer 06 → RMSE: 0.0551, Max Diff: 0.2576
SP 01 TS 02 Layer 07 → RMSE:

In [76]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Prepare dictionary to collect diff_maps for animation
diff_maps_by_layer = {lay: [] for lay in range(nlay)}
animation_titles = []

# Build list of (kstp, kper) pairs (your existing code)
kstpkper_list = []
for kper, (nstp, _, _) in enumerate(perioddata):
    if kper in [0, 1]:  # First and last timestep for SP 1 and 2
        kstpkper_list.append((0, kper))
        kstpkper_list.append((nstp - 1, kper))
    else:  # Only last timestep for others
        kstpkper_list.append((nstp - 1, kper))

# Main loop: process, save PNGs, store diff_maps
for i, (kstp, kper) in enumerate(kstpkper_list):
    sp_num = kper + 1
    ts_num = kstp + 1
    folder_id = f"{i:03d}_sp{sp_num:02d}_ts{ts_num:02d}"
    mf96_folder = os.path.join(mf96_base_dir, folder_id)

    if not os.path.exists(mf96_folder):
        print(f"[Missing Folder] {mf96_folder}")
        continue

    mf6_heads = head_obj.get_data(kstpkper=(kstp, kper))  # shape: (nlay, nrow, ncol)

    for lay in range(nlay):
        mf6_layer = mf6_heads[lay, :, :]
        mf96_file = f"layer_{lay+1}_stressperiod_{sp_num}_timestep_{ts_num}"
        mf96_path = os.path.join(mf96_folder, mf96_file)

        if not os.path.exists(mf96_path):
            print(f"[Missing File] {mf96_path}")
            continue

        try:
            mf96_layer = read_ascii_wrapped_head(mf96_path, (nrow, ncol))
        except Exception as e:
            print(f"[Read Error] {mf96_path}: {e}")
            continue

        nodata = -999.99
        valid_mask = mf96_layer != nodata
        if not np.any(valid_mask):
            print(f"[No Valid Data] {mf96_path}")
            continue

        diff = mf6_layer[valid_mask] - mf96_layer[valid_mask]
        rmse = np.sqrt(np.mean(diff**2))
        max_diff = np.abs(diff).max()

        print(f"SP {sp_num:02d} TS {ts_num:02d} Layer {lay+1:02d} → RMSE: {rmse:.4f}, Max Diff: {max_diff:.4f}")

        diff_map = np.full_like(mf96_layer, np.nan)
        diff_map[valid_mask] = diff

        # Store for animation
        diff_maps_by_layer[lay].append(diff_map)

        # --- Plot and save PNG ---
        plt.figure(figsize=(8, 5))
        cmap = plt.get_cmap("RdBu")  # Blue = +, Red = -
        bounds = np.arange(-1.0, 1.01, 0.1)
        norm = mcolors.BoundaryNorm(bounds, cmap.N)

        im = plt.imshow(diff_map, cmap=cmap, norm=norm)
        plt.colorbar(im, boundaries=bounds, ticks=bounds, extend='both', label="Head Diff (m)")
        plt.title(f"Head Difference (MF6 - MF96)\nSP {sp_num:02d} TS {ts_num:02d} Layer {lay+1:02d}")
        plt.xlabel("Column Index")
        plt.ylabel("Row Index")
        plt.tight_layout()

        png_file = f"sp{sp_num:02d}_ts{ts_num:02d}_layer{lay+1:02d}.png"
        plt.savefig(os.path.join(output_dir, png_file), dpi=300)
        plt.close()

    # Store titles for animation frames (only once per kstp, kper)
    animation_titles.append(f"SP {sp_num:02d} TS {ts_num:02d}")


SP 01 TS 01 Layer 01 → RMSE: 0.0770, Max Diff: 0.2409
SP 01 TS 01 Layer 02 → RMSE: 0.0214, Max Diff: 0.0405
SP 01 TS 01 Layer 03 → RMSE: 0.0250, Max Diff: 0.0760
SP 01 TS 01 Layer 04 → RMSE: 0.0295, Max Diff: 0.0937
SP 01 TS 01 Layer 05 → RMSE: 0.0324, Max Diff: 0.1209
SP 01 TS 01 Layer 06 → RMSE: 0.0507, Max Diff: 0.2607
SP 01 TS 01 Layer 07 → RMSE: 0.0572, Max Diff: 0.4302
SP 01 TS 01 Layer 08 → RMSE: 0.1881, Max Diff: 0.5972
SP 01 TS 01 Layer 09 → RMSE: 0.0572, Max Diff: 0.1523
SP 01 TS 01 Layer 10 → RMSE: 0.0098, Max Diff: 0.0162
SP 01 TS 01 Layer 11 → RMSE: 0.0054, Max Diff: 0.0080
SP 01 TS 01 Layer 12 → RMSE: 0.0016, Max Diff: 0.0018
SP 01 TS 02 Layer 01 → RMSE: 0.0880, Max Diff: 0.2650
SP 01 TS 02 Layer 02 → RMSE: 0.0307, Max Diff: 0.0546
SP 01 TS 02 Layer 03 → RMSE: 0.0338, Max Diff: 0.0898
SP 01 TS 02 Layer 04 → RMSE: 0.0375, Max Diff: 0.1075
SP 01 TS 02 Layer 05 → RMSE: 0.0398, Max Diff: 0.1336
SP 01 TS 02 Layer 06 → RMSE: 0.0551, Max Diff: 0.2576
SP 01 TS 02 Layer 07 → RMSE:

In [ ]:
# create animations by layer
def create_layer_animations(diff_maps_by_layer, animation_titles, output_dir, nlay, fps=2):
    for lay in range(nlay):
        fig, ax = plt.subplots(figsize=(8, 5))
        cmap = plt.get_cmap("RdBu")
        bounds = np.arange(-1.0, 1.01, 0.1)
        norm = mcolors.BoundaryNorm(bounds, cmap.N)

        im = ax.imshow(diff_maps_by_layer[lay][0], cmap=cmap, norm=norm)
        cbar = fig.colorbar(im, boundaries=bounds, ticks=bounds, extend='both', label="Head Diff (m)")
        title = ax.set_title(animation_titles[0] + f" Layer {lay+1:02d} \nHead Difference (MF6 - MF96)")
        ax.set_xlabel("Column Index")
        ax.set_ylabel("Row Index")

        def update(frame):
            im.set_array(diff_maps_by_layer[lay][frame])
            title.set_text(animation_titles[frame] + f" Layer {lay+1:02d} \nHead Difference (MF6 - MF96)")
            return im, title

        ani = animation.FuncAnimation(fig, update, frames=len(diff_maps_by_layer[lay]), blit=False)
        save_path = os.path.join(output_dir, f"layer{lay+1:02d}_head_diff_animation.gif")
        print(f"Saving animation: {save_path}")
        ani.save(save_path, fps=fps, dpi=150)
        plt.close(fig)


In [ ]:
# call animations function
create_layer_animations(diff_maps_by_layer, animation_titles, output_dir, nlay, fps=2)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving animation: heads comparison\layer01_head_diff_animation.gif


MovieWriter ffmpeg unavailable; using Pillow instead.


Saving animation: heads comparison\layer02_head_diff_animation.gif


MovieWriter ffmpeg unavailable; using Pillow instead.


...

Saving animation: heads comparison\layer10_head_diff_animation.gif


MovieWriter ffmpeg unavailable; using Pillow instead.


Saving animation: heads comparison\layer11_head_diff_animation.gif


MovieWriter ffmpeg unavailable; using Pillow instead.


Saving animation: heads comparison\layer12_head_diff_animation.gif
